# Coding Assignment 3: Image-Based Cancer Diagnosis Using Convolutional Neural Networks (CNNs)

# ID 2671507

## section 1- Data ingestion and augment

### Installation cell

In [ ]:
!pip install -q kagglehub

### Block 1: Data Ingestion and Augmentation

In [ ]:
from pathlib import Path
from collections import Counter
import random

import kagglehub
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Config

IMAGE_SIZE = 224
BATCH_SIZE = 32
RANDOM_SEED = 42
NUM_WORKERS = 0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATASET_HANDLE = (
    "aryashah2k/breast-ultrasound-images-dataset"
)

VALID_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff"
}

# Here Binary classification:
# 0 = non-malignant
# 1 = malignant
CLASS_TO_LABEL = {
    "normal": 0,
    "benign": 0,
    "malignant": 1
}

LABEL_TO_NAME = {
    0: "Non-malignant",
    1: "Malignant"
}

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("Device:", DEVICE)

# Download the BUSI dataset

try:
    downloaded_path = kagglehub.dataset_download(
        DATASET_HANDLE
    )

    DATASET_ROOT = Path(downloaded_path)

except Exception as error:
    raise RuntimeError(
        "The BUSI dataset could not be downloaded. "
        "Check your internet connection and Kaggle access."
    ) from error

print("Dataset directory:", DATASET_ROOT)

# Find original BUSI images

def collect_busi_images(root_directory):
    """
    Search recursively for BUSI ultrasound images.

    BUSI contains:
        normal
        benign
        malignant

    Segmentation masks are excluded because their filenames
    contain '_mask'.
    """

    image_paths = []
    labels = []
    original_classes = []

    for file_path in sorted(root_directory.rglob("*")):

        # Ignore directories.
        if not file_path.is_file():
            continue

        # Ignore unsupported file types.
        if file_path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        # Exclude BUSI segmentation masks.
        if "_mask" in file_path.stem.lower():
            continue

        class_name = file_path.parent.name.lower()

        # Ignore images outside the expected class folders
        if class_name not in CLASS_TO_LABEL:
            continue

        image_paths.append(file_path)
        labels.append(CLASS_TO_LABEL[class_name])
        original_classes.append(class_name)

    if len(image_paths) == 0:
        raise RuntimeError(
            f"No BUSI images were found under {root_directory}"
        )

    expected_classes = set(CLASS_TO_LABEL.keys())
    detected_classes = set(original_classes)

    missing_classes = expected_classes - detected_classes

    if missing_classes:
        raise RuntimeError(
            "Missing expected BUSI folders: "
            f"{sorted(missing_classes)}"
        )

    labels = np.asarray(
        labels,
        dtype=np.int64
    )

    return image_paths, labels, original_classes


IMAGE_PATHS, LABELS, ORIGINAL_CLASSES = (
    collect_busi_images(DATASET_ROOT)
)

print("\nDataset information")
print("-" * 45)
print("Number of original images:", len(IMAGE_PATHS))

print(
    "Original BUSI classes:",
    dict(Counter(ORIGINAL_CLASSES))
)

print(
    "Binary classes:",
    {
        LABEL_TO_NAME[label]: count
        for label, count in sorted(
            Counter(LABELS.tolist()).items()
        )
    }
)

# Custom PyTorch Dataset

class BUSIBinaryDataset(Dataset):
    """
    Custom PyTorch dataset for binary classification of
    BUSI ultrasound images.

    Labels:
        0 = normal or benign
        1 = malignant
    """

    def __init__(
        self,
        image_paths,
        labels,
        transform=None
    ):
        if len(image_paths) != len(labels):
            raise ValueError(
                "image_paths and labels must have "
                "the same length."
            )

        self.image_paths = list(image_paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        label = self.labels[index]

        try:
            with Image.open(image_path) as image:
                image = image.convert("RGB")

        except Exception as error:
            raise RuntimeError(
                f"Could not open image: {image_path}"
            ) from error

        if self.transform is not None:
            image = self.transform(image)

        label = torch.tensor(
            label,
            dtype=torch.float32
        )

        return image, label

# Transformation

def build_transforms(dataset_mean, dataset_std):
    """
    Create separate transformations for training and
    evaluation.

    Random augmentation is applied only to training images.
    """

    train_transform = transforms.Compose([
        transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE)
        ),

        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        transforms.RandomVerticalFlip(
            p=0.2
        ),

        transforms.RandomRotation(
            degrees=10
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=dataset_mean,
            std=dataset_std
        )
    ])

    validation_test_transform = transforms.Compose([
        transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE)
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=dataset_mean,
            std=dataset_std
        )
    ])

    return train_transform, validation_test_transform


print("\nBlock 1 completed.")

### Block 2: Normalisation

In [ ]:
@torch.no_grad()
def calculate_mean_std(
    image_paths,
    labels
):
    """
    Calculate channel-wise mean and standard deviation.

    Images are first resized and converted to tensors.
    No augmentation or normalisation is used while
    calculating the statistics.

    Parameters
    ----------
    image_paths:
        Paths of images used to calculate the statistics.

    labels:
        Corresponding image labels.

    Returns
    -------
    dataset_mean:
        List containing the RGB channel means.

    dataset_std:
        List containing the RGB channel standard deviations.
    """

    statistics_transform = transforms.Compose([
        transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE)
        ),

        transforms.ToTensor()
    ])

    statistics_dataset = BUSIBinaryDataset(
        image_paths=image_paths,
        labels=labels,
        transform=statistics_transform
    )

    statistics_loader = DataLoader(
        statistics_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    channel_sum = torch.zeros(
        3,
        dtype=torch.float64
    )

    channel_squared_sum = torch.zeros(
        3,
        dtype=torch.float64
    )

    total_pixels = 0

    for images, _ in statistics_loader:

        images = images.to(torch.float64)

        batch_size = images.shape[0]
        image_height = images.shape[2]
        image_width = images.shape[3]

        channel_sum += images.sum(
            dim=(0, 2, 3)
        )

        channel_squared_sum += (
            images ** 2
        ).sum(
            dim=(0, 2, 3)
        )

        total_pixels += (
            batch_size
            * image_height
            * image_width
        )

    if total_pixels == 0:
        raise RuntimeError(
            "Cannot calculate statistics from "
            "an empty dataset."
        )

    dataset_mean = (
        channel_sum / total_pixels
    )

    dataset_variance = (
        channel_squared_sum / total_pixels
        - dataset_mean ** 2
    )

    # floating-point rounding.
    dataset_variance = torch.clamp(
        dataset_variance,
        min=1e-12
    )

    dataset_std = torch.sqrt(
        dataset_variance
    )

    return (
        dataset_mean.float().tolist(),
        dataset_std.float().tolist()
    )

print("Block 2 completed.")
print(
    "The normalisation function will be called "
    "after creating the training split."
)

### Block 3: Stratified 70/15/15 Partitioning

In [ ]:
from sklearn.model_selection import train_test_split

# Create image indices

all_indices = np.arange(
    len(IMAGE_PATHS)
)

# First split: 70% training and 30% temporary

train_indices, temporary_indices = train_test_split(
    all_indices,
    test_size=0.30,
    stratify=LABELS,
    random_state=RANDOM_SEED
)

# Second split: divide temporary set equally
# 30% temporary becomes: 15% validation and 15% testing

validation_indices, test_indices = train_test_split(
    temporary_indices,
    test_size=0.50,
    stratify=LABELS[temporary_indices],
    random_state=RANDOM_SEED
)

# Ensure that no image appears in multiple partitions

assert set(train_indices).isdisjoint(
    validation_indices
)

assert set(train_indices).isdisjoint(
    test_indices
)

assert set(validation_indices).isdisjoint(
    test_indices
)

# Select paths and labels for a partition

def select_partition(indices):
    partition_paths = [
        IMAGE_PATHS[index]
        for index in indices
    ]

    partition_labels = (
        LABELS[indices].tolist()
    )

    return partition_paths, partition_labels


train_paths, train_labels = select_partition(
    train_indices
)

validation_paths, validation_labels = select_partition(
    validation_indices
)

test_paths, test_labels = select_partition(
    test_indices
)

# Display partition

def print_partition_information(
    partition_name,
    partition_labels,
    total_dataset_size
):
    counts = Counter(partition_labels)

    percentage = (
        len(partition_labels)
        / total_dataset_size
        * 100
    )

    print(f"\n{partition_name}")
    print("-" * 45)
    print("Total images:", len(partition_labels))
    print(f"Dataset percentage: {percentage:.2f}%")

    for label, class_name in LABEL_TO_NAME.items():
        count = counts.get(label, 0)

        class_percentage = (
            count
            / len(partition_labels)
            * 100
        )

        print(
            f"{class_name}: {count} "
            f"({class_percentage:.2f}%)"
        )

total_dataset_size = len(IMAGE_PATHS)

print_partition_information(
    "Training partition",
    train_labels,
    total_dataset_size
)

print_partition_information(
    "Validation partition",
    validation_labels,
    total_dataset_size
)

print_partition_information(
    "Testing partition",
    test_labels,
    total_dataset_size
)

# Calculate mean and standard deviation
# Only the training images are used to prevent leakage.

dataset_mean, dataset_std = calculate_mean_std(
    image_paths=train_paths,
    labels=train_labels
)

print("\nTraining-partition statistics")
print("-" * 45)
print("RGB mean:", dataset_mean)
print("RGB standard deviation:", dataset_std)

# final transformations

train_transform, validation_test_transform = (
    build_transforms(
        dataset_mean=dataset_mean,
        dataset_std=dataset_std
    )
)

# Create final Dataset objects

train_dataset = BUSIBinaryDataset(
    image_paths=train_paths,
    labels=train_labels,
    transform=train_transform
)

validation_dataset = BUSIBinaryDataset(
    image_paths=validation_paths,
    labels=validation_labels,
    transform=validation_test_transform
)

test_dataset = BUSIBinaryDataset(
    image_paths=test_paths,
    labels=test_labels,
    transform=validation_test_transform
)

# Create DataLoaders

loader_generator = torch.Generator()
loader_generator.manual_seed(
    RANDOM_SEED
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    generator=loader_generator
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

# Verify one batch

sample_images, sample_labels = next(
    iter(train_loader)
)


print("\nDataLoader verification")
print("-" * 45)

print(
    "Image batch shape:",
    sample_images.shape
)

print(
    "Label batch shape:",
    sample_labels.shape
)

print(
    "Image data type:",
    sample_images.dtype
)

print(
    "Label data type:",
    sample_labels.dtype
)

print(
    "Minimum normalised value:",
    sample_images.min().item()
)

print(
    "Maximum normalised value:",
    sample_images.max().item()
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

print(
    "Testing batches:",
    len(test_loader)
)

# Expected image batch shape:
# [batch_size, 3, 224, 224]

assert sample_images.ndim == 4
assert sample_images.shape[1] == 3
assert sample_images.shape[2] == IMAGE_SIZE
assert sample_images.shape[3] == IMAGE_SIZE

print("\nsection A completed successfully.")

## Phase 2 : CNN Architecture and Training

### Block 1: CNN Architecture



In [ ]:
import torch
import torch.nn as nn

class CancerCNN(nn.Module):
    """
    Custom CNN for binary cancer diagnosis.

    Input shape:
        [batch_size, 3, 224, 224]

    Output shape:
        [batch_size, 1]

    Output meaning:
        Probability of the malignant class.
    """

    def __init__(self):
        super().__init__()

        # Feature extraction layers

        self.feature_extractor = nn.Sequential(

            # Convolutional block 1
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # Convolutional block 2
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # Convolutional block 3
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=2,
                stride=2
            ),

            # Produce a fixed feature-map size
            nn.AdaptiveAvgPool2d((4, 4))
        )

        # Classification head

        self.classifier = nn.Sequential(

            # Flatten the final feature maps
            nn.Flatten(),

            # Fully connected hidden layer
            nn.Linear(
                in_features=128 * 4 * 4,
                out_features=256
            ),
            nn.ReLU(),

            # Regularisation
            nn.Dropout(p=0.5),

            # Single output node
            nn.Linear(
                in_features=256,
                out_features=1
            ),

            # Convert output to a probability
            nn.Sigmoid()
        )

    def forward(self, images):
        features = self.feature_extractor(images)
        probabilities = self.classifier(features)

        return probabilities

# Creating the CNN
model = CancerCNN().to(DEVICE)

print(model)
print("\nModel device:", next(model.parameters()).device)

# Verify the model using one batch

sample_images, sample_labels = next(
    iter(train_loader)
)

sample_images = sample_images.to(DEVICE)

with torch.no_grad():
    sample_outputs = model(sample_images)

print("Input batch shape:", sample_images.shape)
print("Output batch shape:", sample_outputs.shape)
print(
    "Output probability range:",
    sample_outputs.min().item(),
    "to",
    sample_outputs.max().item()
)

### Block 2: Optimisation and Training

In [ ]:
import time
import torch.optim as optim

# Training configuration

NUM_EPOCHS = 21
LEARNING_RATE = 0.001
BEST_MODEL_PATH = "best_cancer_cnn.pth"

# Binary Cross-Entropy loss
criterion = nn.BCELoss()

# Adam optimiser
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# loss and accuracy for every epoch


history = {
    "train_loss": [],
    "validation_loss": [],
    "train_accuracy": [],
    "validation_accuracy": []
}

# Training function

def train_one_epoch(
    model,
    data_loader,
    criterion,
    optimizer,
    device
):
    """
    Train the CNN for one epoch.

    Returns:
        average_loss
        accuracy
    """

    # Enable training behaviour for BatchNorm and Dropout
    model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in data_loader:

        # Move data to CPU or GPU
        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        ).float().view(-1)

        # Clear gradients from the previous batch
        optimizer.zero_grad()

        # Forward pass
        probabilities = model(
            images
        ).view(-1)

        # Calculate loss
        loss = criterion(
            probabilities,
            labels
        )

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        batch_size = images.size(0)

        # Accumulate batch loss
        running_loss += (
            loss.item() * batch_size
        )

        # Convert probabilities into class predictions
        predictions = (
            probabilities >= 0.5
        ).float()

        correct_predictions += (
            predictions == labels
        ).sum().item()

        total_samples += batch_size

    average_loss = (
        running_loss / total_samples
    )

    accuracy = (
        correct_predictions / total_samples
    )

    return average_loss, accuracy

# Validation function

@torch.no_grad()
def validate_one_epoch(
    model,
    data_loader,
    criterion,
    device
):
    """
    Evaluate the CNN on the validation partition.

    No gradients or weight updates are performed.

    Returns:
        average_loss
        accuracy
    """

    # Disable training behaviour for BatchNorm and Dropout
    model.eval()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in data_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        ).float().view(-1)

        # Forward pass
        probabilities = model(
            images
        ).view(-1)

        # Calculate validation loss
        loss = criterion(
            probabilities,
            labels
        )

        batch_size = images.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        predictions = (
            probabilities >= 0.5
        ).float()

        correct_predictions += (
            predictions == labels
        ).sum().item()

        total_samples += batch_size

    average_loss = (
        running_loss / total_samples
    )

    accuracy = (
        correct_predictions / total_samples
    )

    return average_loss, accuracy

# Train the model

best_validation_loss = float("inf")
best_epoch = 0

training_start_time = time.time()


for epoch in range(1, NUM_EPOCHS + 1):

    # Train for one epoch
    train_loss, train_accuracy = train_one_epoch(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE
    )

    # Validate after the epoch
    validation_loss, validation_accuracy = (
        validate_one_epoch(
            model=model,
            data_loader=validation_loader,
            criterion=criterion,
            device=DEVICE
        )
    )

    # Save history for sectionc C learning curves
    history["train_loss"].append(
        train_loss
    )

    history["validation_loss"].append(
        validation_loss
    )

    history["train_accuracy"].append(
        train_accuracy
    )

    history["validation_accuracy"].append(
        validation_accuracy
    )

    # Save the model with the lowest validation loss
    if validation_loss < best_validation_loss:

        best_validation_loss = validation_loss
        best_epoch = epoch

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        save_message = " | Best model saved"

    else:
        save_message = ""

    # Display results for the current epoch
    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: "
        f"{train_accuracy * 100:.2f}% | "
        f"Validation Loss: "
        f"{validation_loss:.4f} | "
        f"Validation Accuracy: "
        f"{validation_accuracy * 100:.2f}%"
        f"{save_message}"
    )


training_duration = time.time() - training_start_time

# Training summary

print("\nTraining completed.")
print("Best epoch:", best_epoch)

print(
    "Best validation loss:",
    f"{best_validation_loss:.4f}"
)

print(
    "Training duration:",
    f"{training_duration / 60:.2f} minutes"
)

print(
    "Best model saved at:",
    BEST_MODEL_PATH
)

## Section 3 - Deep Learning Diagnostics & Visualization (Matplotlib)

### Block 1: Learning Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

epochs = np.arange(
    1,
    len(history["train_loss"]) + 1
)

# Create two plots side by side
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

# Loss curve

axes[0].plot(
    epochs,
    history["train_loss"],
    label="Training Loss"
)

axes[0].plot(
    epochs,
    history["validation_loss"],
    label="Validation Loss"
)

axes[0].set_title("Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Accuracy curve

train_accuracy_percent = [
    accuracy * 100
    for accuracy in history["train_accuracy"]
]

validation_accuracy_percent = [
    accuracy * 100
    for accuracy in history["validation_accuracy"]
]

axes[1].plot(
    epochs,
    train_accuracy_percent,
    label="Training Accuracy"
)

axes[1].plot(
    epochs,
    validation_accuracy_percent,
    label="Validation Accuracy"
)

axes[1].set_title(
    "Training and Validation Accuracy"
)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()

plt.savefig(
    "learning_curves.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Learning curves saved as learning_curves.png")

### Block 2: ROC Curve and AUC

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    roc_auc_score
)
-
# Load the best saved model

try:
    model.load_state_dict(
        torch.load(
            BEST_MODEL_PATH,
            map_location=DEVICE,
            weights_only=True
        )
    )

except TypeError:
    # Compatibility with older PyTorch versions
    model.load_state_dict(
        torch.load(
            BEST_MODEL_PATH,
            map_location=DEVICE
        )
    )

model = model.to(DEVICE)
model.eval()

# Collect test labels and probabilities

test_true_labels = []
test_probabilities = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(DEVICE)

        probabilities = model(
            images
        ).view(-1)

        test_true_labels.extend(
            labels.cpu().numpy()
        )

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )

test_true_labels = np.array(
    test_true_labels,
    dtype=int
)

test_probabilities = np.array(
    test_probabilities
)

# Calculate ROC and AUC

false_positive_rate, true_positive_rate, thresholds = (
    roc_curve(
        test_true_labels,
        test_probabilities
    )
)

auc_score = roc_auc_score(
    test_true_labels,
    test_probabilities
)

# Plot ROC curve

plt.figure(figsize=(7, 6))

plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=f"CNN AUC = {auc_score:.4f}"
)

# Random-classifier reference line
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.title("Receiver Operating Characteristic Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)

plt.savefig(
    "roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Test AUC score: {auc_score:.4f}")
print("ROC curve saved as roc_curve.png")

### Block 3: Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Convert probabilities into binary predictions

test_predictions = (
    test_probabilities >= 0.5
).astype(int)

# Calculate confusion matrix

confusion_matrix_values = confusion_matrix(
    test_true_labels,
    test_predictions
)

# Display and save confusion matrix

display = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_values,
    display_labels=[
        "Non-malignant",
        "Malignant"
    ]
)

fig, ax = plt.subplots(
    figsize=(7, 6)
)

display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d"
)

ax.set_title("Test Set Confusion Matrix")

plt.savefig(
    "confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Extract confusion-matrix values

true_negative, false_positive, false_negative, true_positive = (
    confusion_matrix_values.ravel()
)


test_accuracy = (
    true_positive + true_negative
) / confusion_matrix_values.sum()

print("Confusion matrix:")
print(confusion_matrix_values)

print("\nTest results")
print("-" * 40)
print("True negatives:", true_negative)
print("False positives:", false_positive)
print("False negatives:", false_negative)
print("True positives:", true_positive)
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

print(
    "Confusion matrix saved as "
    "confusion_matrix.png"
)